In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
import os

# --- 1. DATA SETTINGS ---
img_size = 112
batch_size = 8 

# Load your personal face photos (Specialization Dataset)
train_ds = tf.keras.utils.image_dataset_from_directory(
    './finetuning',
    label_mode='categorical',
    image_size=(img_size, img_size),
    batch_size=batch_size,
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    './finetuningtest',
    label_mode='categorical',
    image_size=(img_size, img_size),
    batch_size=batch_size
)

# --- 2. ARCHITECTURE (Must match your Keras 2 Training exactly) ---
def se_block(input_tensor, ratio=8):
    filters = input_tensor.shape[-1]
    se = layers.GlobalAveragePooling2D()(input_tensor)
    se = layers.Reshape((1, 1, filters))(se)
    se = layers.Dense(filters // ratio, activation='relu', kernel_initializer='he_normal', use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', kernel_initializer='he_normal', use_bias=False)(se)
    return layers.Multiply()([input_tensor, se])

def res_block(x, filters):
    shortcut = x
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, padding='same')(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Conv2D(filters, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x) 
    
    x = layers.Conv2D(filters, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = se_block(x) 
    
    x = layers.Add()([x, shortcut])
    return layers.Activation('swish')(x)

# Build Model
inputs = layers.Input(shape=(img_size, img_size, 3))
x = layers.Rescaling(1./255)(inputs) 

x = layers.Conv2D(64, 3, padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('swish')(x)

x = res_block(x, 64)
x = layers.Conv2D(128, 3, strides=2, padding='same')(x)
x = layers.BatchNormalization()(x)
x = res_block(x, 128)

x = layers.Conv2D(256, 3, strides=2, padding='same')(x)
x = layers.BatchNormalization()(x)
x = res_block(x, 256)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation='swish')(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(7, activation='softmax')(x) 

model = models.Model(inputs, outputs)

# --- 3. INJECT THE K2 WEIGHTS ---
# This loads the 69% accuracy weights from your RAF-DB K2 run
model.load_weights('cnn_emotion_k2_final.h5') 
print("✅ RAF-DB Knowledge injected successfully.")

# --- 4. GENTLE FINE-TUNING ---
# Using a very low learning rate to avoid "forgetting" the base emotions
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("🚀 Specializing model on your face...")
history = model.fit(
    train_ds, 
    validation_data=val_ds, 
    epochs=30
)

# --- 5. EXPORT EVERYTHING ---
# Save the final specialized model for the Ensemble
model.save('cnn_specialized_k2.h5')

# Convert to TFLite for your Flutter App
print("Converting to TFLite...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('final_specialized_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("✅ Success! 'cnn_specialized_k2.h5' (Ensemble) and 'final_specialized_model.tflite' (App) are ready.")


Found 361 files belonging to 7 classes.
Found 157 files belonging to 7 classes.


✅ RAF-DB Knowledge injected successfully.
🚀 Specializing model on your face...
Epoch 1/30


46/46 [==============================] - 150s 3s/step - loss: 2.0033 - accuracy: 0.2410 - val_loss: 2.0047 - val_accuracy: 0.1975
Epoch 2/30
46/46 [==============================] - 3148s 70s/step - loss: 1.6285 - accuracy: 0.3767 - val_loss: 1.7054 - val_accuracy: 0.3312
Epoch 3/30
46/46 [==============================] - 73s 2s/step - loss: 1.4375 - accuracy: 0.4404 - val_loss: 1.4482 - val_accuracy: 0.4013
Epoch 4/30
46/46 [==============================] - 73s 2s/step - loss: 1.3132 - accuracy: 0.5291 - val_loss: 1.2478 - val_accuracy: 0.4904
Epoch 5/30
46/46 [==============================] - 72s 2s/step - loss: 1.2080 - accuracy: 0.5623 - val_loss: 1.1645 - val_accuracy: 0.5414
Epoch 6/30
46/46 [==============================] - 74s 2s/step - loss: 1.1380 - accuracy: 0.6150 - val_loss: 1.0448 - val_accuracy:

C:\Users\ASUS\anaconda3\envs\fyp_train_k2\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Converting to TFLite...
INFO:tensorflow:Assets written to: C:\Users\ASUS\AppData\Local\Temp\tmpy229hsi6\assets


INFO:tensorflow:Assets written to: C:\Users\ASUS\AppData\Local\Temp\tmpy229hsi6\assets


✅ Success! 'cnn_specialized_k2.h5' (Ensemble) and 'final_specialized_model.tflite' (App) are ready.
